# 유튜브 댓글 데이터 적응적 전처리 파이프라인

이 노트북은 `Kiwi` 형태소 분석기를 사용하여 유튜브 댓글 데이터를 정제합니다.

## 주요 기능
1. **동사/형용사 원형 복원**: 어간 뒤에 `-다`를 붙여 표준형으로 변환 (예: `힘들` -> `힘들다`)
2. **유튜브 플랫폼 불용어 제거**: `좋아요`, `구독`, `댓글` 등 매체 특성 단어 제거
3. **저빈도 단어 필터링**: 전체 빈도 10회 이하 단어 제거
4. **단문 제거**: 유효 토큰이 1개 이하인 댓글 제외

In [1]:
import pandas as pd
from kiwipiepy import Kiwi
from kiwipiepy.utils import Stopwords
from tqdm import tqdm
from collections import Counter
import os

# 1. 데이터 로드
input_file = os.path.join("..", "data", "processed", "토크나이징_전_전처리.csv")
df = pd.read_csv(input_file, encoding='utf-8-sig')
df['cleaned_text'] = df['cleaned_text'].fillna("")
print(f"Total rows: {len(df)}")

Total rows: 74356


C:\Users\user\AppData\Local\Temp\ipykernel_13560\2721558898.py:10: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file, encoding='utf-8-sig')


In [2]:
# 2. Kiwi 초기화 및 불용어 설정
kiwi = Kiwi()
stopwords_kiwi = Stopwords()

# 기본 불용어 (어간과 원형 모두 추가)
basic_stopwords = ['하다', '있다', '되다', '않다', '같다', '싶다', '이렇다', '저렇다', '어떻다', '나오다', '가지다', '보다', '해주다', '그렇다', '알다', '모르다', '생각하다', '말하다', '받다', '나다', '오다', '가다', '들다']
for word in basic_stopwords:
    for tag in ['VV', 'VA', 'VX']:
        stopwords_kiwi.add((word, tag))
        if word.endswith('다'):
            stopwords_kiwi.add((word[:-1], tag))
    
# 유튜브 플랫폼 관련 불용어
platform_words = ['유튜브', '영상', '댓글', '구독', '채널', '조회수', '알림', '설정', '동영상', '업로드', '수정', '삭제', '링크', '주소', '사이트', '클릭']
for word in platform_words:
    for tag in ['NNG', 'NNP', 'VV', 'VA']:
        stopwords_kiwi.add((word, tag))
        if word.endswith('다'):
            stopwords_kiwi.add((word[:-1], tag))

In [9]:
# 3. 형태소 분석 및 '-다' 부착 로직
def custom_tokenize(text):
    if not text:
        return []
    
    tokens = kiwi.tokenize(text, stopwords=stopwords_kiwi, normalize_coda=True)
    
    result = []
    for token in tokens:
        if token.tag in ['NNG', 'NNP', 'VV', 'VA']:
            form = token.form
            
            # '좋', '좋다'를 건너뛰는 조건문 삭제 완료
            
            # 동사(VV)나 형용사(VA)인 경우 '다'를 붙임
            if token.tag in ['VV', 'VA']:
                form += "다"
                
            # 길이가 2 이상인 형태소만 결과에 추가 ('좋'은 위에서 '좋다'가 되므로 2글자가 되어 포함됨)
            if len(form) >= 2:
                result.append(form)
    return result

print("Analyzing morphs and lemmatizing...")
tqdm.pandas()
df['tokens'] = df['cleaned_text'].progress_apply(custom_tokenize)

Analyzing morphs and lemmatizing...


100%|██████████| 74356/74356 [02:10<00:00, 570.55it/s] 


In [10]:
# 4. 저빈도 단어 제거
print("Calculating frequencies...")
all_tokens = [token for sublist in df['tokens'] for token in sublist]
token_counts = Counter(all_tokens)

threshold = 10
low_freq_tokens = {word for word, count in token_counts.items() if count <= threshold}
df['filtered_tokens'] = df['tokens'].apply(lambda x: [t for t in x if t not in low_freq_tokens])

# 5. 단일 토큰 제거
df_final = df[df['filtered_tokens'].apply(len) > 1].copy()
df_final['final_text'] = df_final['filtered_tokens'].apply(lambda x: " ".join(x))

print(f"Final rows: {len(df_final)}")

Calculating frequencies...
Final rows: 55679


In [11]:
# 6. 결과 저장
output_dir = os.path.join("..", "data", "processed")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "최종정제_v2.csv")
df_final[['cid', 'final_text']].to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"Saved to {output_path}")

Saved to ..\data\processed\최종정제_v2.csv
